In [2]:
print("NBA Sports Gambling project is working!")

NBA Sports Gambling project is working!


In [3]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib
import seaborn as sns

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Seaborn:", sns.__version__)

Pandas: 3.0.6
NumPy: 2.5.3
Scikit-learn: 1.9.1
Matplotlib: 3.11.2
Seaborn: 0.13.2


# NBA Sports Gambling — Data Exploration

## Project Goal

Predict the total number of points scored by both teams in an NBA game.

The original paper focuses on the Over/Under betting indicator.

## Original Dataset Sources

1. Basketball Reference
   - NBA game/team/player statistics
   - Retrieved using Fran Goitia's NBA crawler

2. Sportsbook Review Online
   - Historical NBA betting odds
   - Includes Over/Under lines

## Feature Strategy

For each NBA matchup, use information from each team's previous three games.

Important features include:
- Points Scored
- Points Scored Against
- Total Rebounds
- Offensive Rating
- Plus/Minus
- Opponent season averages
- Days since last game
- Travel distance

## Models to Reproduce

- Random Forest
- Collaborative Filtering
- Neural Network
- LSTM

## Additional Model

- XGBoost

In [4]:
import pandas as pd

games = pd.read_csv("../data/raw/games.csv")

games.head()

,GAME_DATE_EST,GAME_ID,GAME_STATUS_TEXT,HOME_TEAM_ID,VISITOR_TEAM_ID,SEASON,TEAM_ID_home,PTS_home,FG_PCT_home,FT_PCT_home,...,AST_home,REB_home,TEAM_ID_away,PTS_away,FG_PCT_away,FT_PCT_away,FG3_PCT_away,AST_away,REB_away,HOME_TEAM_WINS
0,2025-02-12,22400769,Final,1610612738,1610612759,2024,1610612738,116.0,0.462,0.667,...,25.0,56.0,1610612759,103.0,0.438,0.833,0.231,30.0,38.0,1
1,2025-02-12,22400770,Final,1610612753,1610612766,2024,1610612753,102.0,0.411,0.737,...,22.0,52.0,1610612766,86.0,0.395,0.933,0.308,21.0,43.0,1
2,2025-02-12,22400771,Final,1610612764,1610612754,2024,1610612764,130.0,0.485,0.813,...,29.0,46.0,1610612754,134.0,0.467,0.900,0.457,38.0,42.0,0
3,2025-02-12,22400772,Final,1610612751,1610612755,2024,1610612751,100.0,0.471,0.800,...,23.0,50.0,1610612755,96.0,0.432,0.818,0.242,25.0,38.0,1
4,2025-02-12,22400773,Final,1610612752,1610612737,2024,1610612752,149.0,0.574,0.811,...,28.0,39.0,1610612737,148.0,0.500,0.686,0.368,43.0,47.0,1


In [6]:
games.shape

(29755, 21)

In [7]:
games.columns.tolist()

['GAME_DATE_EST',
 'GAME_ID',
 'GAME_STATUS_TEXT',
 'HOME_TEAM_ID',
 'VISITOR_TEAM_ID',
 'SEASON',
 'TEAM_ID_home',
 'PTS_home',
 'FG_PCT_home',
 'FT_PCT_home',
 'FG3_PCT_home',
 'AST_home',
 'REB_home',
 'TEAM_ID_away',
 'PTS_away',
 'FG_PCT_away',
 'FT_PCT_away',
 'FG3_PCT_away',
 'AST_away',
 'REB_away',
 'HOME_TEAM_WINS']

In [8]:
games.info()

<class 'pandas.DataFrame'>
RangeIndex: 29755 entries, 0 to 29754
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   GAME_DATE_EST     29755 non-null  str    
 1   GAME_ID           29755 non-null  int64  
 2   GAME_STATUS_TEXT  29755 non-null  str    
 3   HOME_TEAM_ID      29755 non-null  int64  
 4   VISITOR_TEAM_ID   29755 non-null  int64  
 5   SEASON            29755 non-null  int64  
 6   TEAM_ID_home      29755 non-null  int64  
 7   PTS_home          29656 non-null  float64
 8   FG_PCT_home       29656 non-null  float64
 9   FT_PCT_home       29656 non-null  float64
 10  FG3_PCT_home      29656 non-null  float64
 11  AST_home          29656 non-null  float64
 12  REB_home          29656 non-null  float64
 13  TEAM_ID_away      29755 non-null  int64  
 14  PTS_away          29656 non-null  float64
 15  FG_PCT_away       29656 non-null  float64
 16  FT_PCT_away       29656 non-null  float64
 17  FG3_

In [9]:
games["SEASON"].unique()

array([2024, 2023, 2022, 2021, 2020, 2019, 2013, 2012, 2011, 2010, 2009,
       2008, 2007, 2006, 2005, 2004, 2003, 2018, 2017, 2016, 2015, 2014])

In [10]:
games["SEASON"].min(), games["SEASON"].max()

(np.int64(2003), np.int64(2024))

In [11]:
games["GAME_DATE_EST"].min(), games["GAME_DATE_EST"].max()

('2003-10-05', '2025-02-12')

In [12]:
games.isnull().sum()

GAME_DATE_EST        0
GAME_ID              0
GAME_STATUS_TEXT     0
HOME_TEAM_ID         0
VISITOR_TEAM_ID      0
SEASON               0
TEAM_ID_home         0
PTS_home            99
FG_PCT_home         99
FT_PCT_home         99
FG3_PCT_home        99
AST_home            99
REB_home            99
TEAM_ID_away         0
PTS_away            99
FG_PCT_away         99
FT_PCT_away         99
FG3_PCT_away        99
AST_away            99
REB_away            99
HOME_TEAM_WINS       0
dtype: int64

In [13]:
games["GAME_ID"].duplicated().sum()

np.int64(29)

In [14]:
games[[
    "GAME_DATE_EST",
    "SEASON",
    "TEAM_ID_home",
    "PTS_home",
    "TEAM_ID_away",
    "PTS_away"
]].head(10)

,GAME_DATE_EST,SEASON,TEAM_ID_home,PTS_home,TEAM_ID_away,PTS_away
0,2025-02-12,2024,1610612738,116.0,1610612759,103.0
1,2025-02-12,2024,1610612753,102.0,1610612766,86.0
2,2025-02-12,2024,1610612764,130.0,1610612754,134.0
3,2025-02-12,2024,1610612751,100.0,1610612755,96.0
4,2025-02-12,2024,1610612752,149.0,1610612737,148.0
5,2025-02-12,2024,1610612761,108.0,1610612739,131.0
6,2025-02-12,2024,1610612741,110.0,1610612765,128.0
7,2025-02-12,2024,1610612750,101.0,1610612749,103.0
8,2025-02-12,2024,1610612740,111.0,1610612758,119.0
9,2025-02-12,2024,1610612760,115.0,1610612748,101.0
